# Predicting Annual Income from Census Data
**Supervised Binary Classification & End-to-End Machine Learning Pipeline**
*Academic Demonstration on 1994 U.S. Census Bureau Database (Barry Becker & Ronny Kohavi, UCI ML Repository)*

---
### Academic & Ethical Usage Disclaimer
> **IMPORTANT:** This project demonstrates machine learning methodologies on historical 1994 census data. Models trained on this dataset reflect historical socioeconomic structures of the mid-1990s and must **never** be used for real-world automated decision-making regarding hiring, credit scoring, insurance, criminal justice, or welfare benefits.


In [ ]:
import sys
from pathlib import Path
import warnings
import json
warnings.filterwarnings('ignore')

# Add project root to sys.path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
if str(PROJECT_ROOT.parent) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT.parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from src.data_preprocessing import (
    load_and_clean_data,
    inspect_raw_data,
    split_data,
    create_full_pipeline,
    POSITIVE_CLASS,
    NEGATIVE_CLASS,
)


## 1. Raw Dataset Inspection
We inspect the dataset dimensions, missing values, string anomalies, question mark placeholders (`?`), and duplicate observations before applying any transformations.


In [ ]:
raw_stats = inspect_raw_data()
print(f"Raw Dataset Shape: {raw_stats['raw_shape']}")
print(f"Duplicate Rows in Raw Data: {raw_stats['raw_duplicates']}")
print("Question Mark ('?') Counts:", raw_stats['question_mark_counts'])
print("Raw Target Distribution:", raw_stats['raw_target_distribution'])


## 2. Controlled Data Cleaning and Preprocessing
The preprocessing follows strict leak-free standards:
1. Strip leading and trailing whitespace from string variables.
2. Convert placeholder `?` symbols to `NaN`.
3. Normalize target variable `income` by stripping trailing periods, strictly asserting binary classes: `<=50K` and `>50K`.
4. Remove identical duplicate observations (52 duplicate rows removed).
5. **Feature Engineering Note on `education` vs `education-num`:** `education` and `education-num` have a strict 1-to-1 bijection (Preschool=1 to Doctorate=16). To avoid double-weighting, `education-num` is retained as an ordinal numerical feature.
6. **Feature Note on `fnlwgt`:** `fnlwgt` represents the Census population sampling weight, not an individual characteristic. It has near-zero correlation with income (r = -0.006) and is excluded from predictors.


In [ ]:
df_clean, report = load_and_clean_data()
print("Cleaning Summary Report:")
for k, v in report.items():
    print(f"  {k}: {v}")

print("\nPost-Cleaning Target Counts:")
print(df_clean['income'].value_counts())
print("\nPost-Cleaning Target Proportions:")
print(df_clean['income'].value_counts(normalize=True))


## 3. Exploratory Data Analysis & Mandatory Figures (1-8)
We examine the distribution of income, demographic variables, missing data patterns, capital gain/loss zero-inflation, and numeric feature correlations.


In [ ]:
from IPython.display import Image, display

figures = [
    ("Fig 1: Income Class Distribution", "reports/figures/01_income_class_distribution.png"),
    ("Fig 2: Missing Values per Feature", "reports/figures/02_missing_values_per_feature.png"),
    ("Fig 3: Age Distribution by Income", "reports/figures/03_age_distribution_by_income.png"),
    ("Fig 4: Education Level by Income", "reports/figures/04_education_level_by_income.png"),
    ("Fig 5: Occupation by Income", "reports/figures/05_occupation_by_income.png"),
    ("Fig 6: Hours per Week by Income", "reports/figures/06_hours_per_week_by_income.png"),
    ("Fig 7: Capital Gain/Loss Distribution", "reports/figures/07_capital_gain_loss_distribution.png"),
    ("Fig 8: Numeric Correlation Heatmap", "reports/figures/08_numeric_correlation_heatmap.png"),
]

for title, path in figures:
    fig_p = Path(path)
    if not fig_p.exists():
        fig_p = Path("../" + path)
    if fig_p.exists():
        print(f"=== {title} ===")
        display(Image(filename=str(fig_p), width=650))


## 4. Leak-Free Preprocessing Pipeline & Data Splitting
The dataset is split into training (80%) and testing (20%) using stratified sampling on `income` with `random_state=42`. All imputation (median for numeric, mode for categorical), scaling (`StandardScaler`), and encoding (`OneHotEncoder(handle_unknown='ignore')`) are encapsulated inside a scikit-learn `Pipeline` and fit only on training folds.


In [ ]:
X_train, X_test, y_train, y_test = split_data(df_clean, test_size=0.2, random_state=42)
print(f"X_train Shape: {X_train.shape}, y_train Shape: {y_train.shape}")
print(f"X_test Shape: {X_test.shape}, y_test Shape: {y_test.shape}")
print("\nTraining Set Income Proportions:\n", y_train.value_counts(normalize=True))
print("\nTest Set Income Proportions:\n", y_test.value_counts(normalize=True))


## 5. Model Cross-Validation and Class Imbalance Comparison
We evaluate multiple classifier families using 5-fold Stratified Cross-Validation on the training set:
- Baseline: `DummyClassifier (Most Frequent)`
- Linear: `Logistic Regression` (Default vs Balanced class weight)
- Tree: `Decision Tree` (Default vs Balanced class weight)
- Ensemble: `Random Forest` (Default vs Balanced class weight)
- Boosting: `HistGradientBoosting` (Default vs Balanced class weight)


In [ ]:
cv_path = Path("reports/cv_model_comparison.csv")
if not cv_path.exists():
    cv_path = Path("../reports/cv_model_comparison.csv")
cv_df = pd.read_csv(cv_path)
cv_df.style.format(precision=4)


## 6. Hyperparameter Tuning & Pipeline Serialization
`HistGradientBoostingClassifier` emerged as the strongest performer. We optimized hyperparameters (`learning_rate`, `max_iter`, `max_leaf_nodes`, `l2_regularization`) via `GridSearchCV` on the training folds using average precision (PR-AUC) as the optimization objective.


In [ ]:
meta_path = Path("models/best_model_metadata.json")
if not meta_path.exists():
    meta_path = Path("../models/best_model_metadata.json")
with open(meta_path) as f:
    meta = json.load(f)
print("Tuned Model Metadata:")
print(json.dumps(meta, indent=2))


## 7. Final Evaluation on Untouched Test Set (Figures 9-14)
We evaluate the tuned pipeline on the test set (N = 9,758) exactly once. The positive class is explicitly defined as `>50K`.


In [ ]:
comp_path = Path("reports/model_comparison.csv")
if not comp_path.exists():
    comp_path = Path("../reports/model_comparison.csv")
test_comp = pd.read_csv(comp_path)
test_comp.style.format(precision=4)


In [ ]:
eval_figures = [
    ("Fig 9: Model Metrics Comparison", "reports/figures/09_model_metrics_comparison.png"),
    ("Fig 10: Best Model Confusion Matrix", "reports/figures/10_best_model_confusion_matrix.png"),
    ("Fig 11: ROC Curves", "reports/figures/11_roc_curves.png"),
    ("Fig 12: Precision-Recall Curves", "reports/figures/12_precision_recall_curves.png"),
    ("Fig 13: Permutation Feature Importance", "reports/figures/13_feature_importance.png"),
    ("Fig 14: Fairness Audit by Sex and Race", "reports/figures/14_fairness_by_sex_and_race.png"),
]

for title, path in eval_figures:
    fig_p = Path(path)
    if not fig_p.exists():
        fig_p = Path("../" + path)
    if fig_p.exists():
        print(f"=== {title} ===")
        display(Image(filename=str(fig_p), width=650))


## 8. Permutation Feature Importance & Model Interpretability
Feature contribution is computed using permutation importance on the untouched test set, measuring decrease in PR-AUC.


In [ ]:
feat_path = Path("reports/feature_importance.csv")
if not feat_path.exists():
    feat_path = Path("../reports/feature_importance.csv")
feat_df = pd.read_csv(feat_path)
print("Top Permutation Feature Importances:")
print(feat_df.head(10).to_string(index=False))


## 9. Fairness & Demographic Disparity Audit
We conduct a rigorous audit across demographic groups (`sex` and `race`), comparing the full model against an ablation model trained without `sex` and `race`.


In [ ]:
fair_path = Path("reports/fairness_report.csv")
if not fair_path.exists():
    fair_path = Path("../reports/fairness_report.csv")
fair_df = pd.read_csv(fair_path)
fair_df.style.format(precision=4)


### Fairness Audit Findings:
1. **Historical Disparities in Data:** In the 1994 census data, the positive rate (>50K) for females was ~11% vs ~31% for males. The model reflects this base rate, yielding a Selection Rate of 7.88% for females vs 27.83% for males.
2. **The Proxy Variable Effect:** When `sex` and `race` are removed from model training, the female selection rate changes only trivially from 7.88% to 7.97%, and male selection rate from 27.83% to 27.55%. Features like `marital-status` (Married-civ-spouse), `relationship` (Husband vs Wife), `occupation`, and `hours-per-week` act as strong proxies.
3. **Conclusion:** Removing sensitive variables does **not** solve algorithmic bias. Automated income prediction must not be deployed for critical individual decisions.


## 10. Sample Test Predictions & Pipeline Inference
We verify that the saved `.joblib` pipeline can be loaded and executed on sample observations.


In [ ]:
sample_path = Path("reports/sample_predictions.csv")
if not sample_path.exists():
    sample_path = Path("../reports/sample_predictions.csv")
sample_df = pd.read_csv(sample_path)
sample_df[['age', 'education-num', 'marital-status', 'occupation', 'hours-per-week', 'actual_income', 'predicted_income', 'predicted_probability_>50K']].head(10)
